# SQL + Python Basic Data Analysis
This notebook demonstrates basic data analysis using **MySQL + Python (pandas)** on three tables:
- contracts
- billing_histories
- contract_sign_up_details


## 1. Install Required Libraries

In [35]:
import pandas as pd
import numpy as np
import mysql.connector

## 2. Connect to MySQL Database

In [36]:
activate = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Obito@11',
    database='mozaic_db'
)


df_test = pd.read_sql('SELECT * FROM contracts LIMIT 5', activate)
df_test

C:\Users\Logeshwaran Sekar\AppData\Local\Temp\ipykernel_4936\2824030542.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_test = pd.read_sql('SELECT * FROM contracts LIMIT 5', activate)


,id,created_at,terminated_at,state,product_identifier,payment_provider_config_profile_id,product_type,signed_at,billing_histories_count,billing_histories_sum_in_euro_cents
0,264929473,2024-10-01 00:08:55,2024-12-02 01:09:26,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:09:10,None,None
1,264929559,2024-10-01 00:13:23,2025-01-01 01:13:51,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:13:42,None,None
2,264929588,2024-10-01 00:14:44,2024-11-12 01:16:14,terminated,PlStreamingPflBellatorWire,173,Wire,2024-10-01 00:16:06,None,None
3,264929939,2024-10-01 00:32:06,2024-10-02 00:35:56,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:32:24,None,None
4,264930257,2024-10-01 00:48:45,2025-01-01 01:49:11,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:49:07,None,None


## 3. Load Tables into DataFrames

In [37]:
contracts = pd.read_sql('SELECT * FROM contracts', activate)
billing_histories = pd.read_sql('SELECT * FROM billing_histories', activate)
contract_signup_details = pd.read_sql('SELECT * FROM contract_sign_up_details', activate)

contracts.head()

C:\Users\Logeshwaran Sekar\AppData\Local\Temp\ipykernel_4936\1842764635.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contracts = pd.read_sql('SELECT * FROM contracts', activate)
C:\Users\Logeshwaran Sekar\AppData\Local\Temp\ipykernel_4936\1842764635.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  billing_histories = pd.read_sql('SELECT * FROM billing_histories', activate)
C:\Users\Logeshwaran Sekar\AppData\Local\Temp\ipykernel_4936\1842764635.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contract_signup_det

,id,created_at,terminated_at,state,product_identifier,payment_provider_config_profile_id,product_type,signed_at,billing_histories_count,billing_histories_sum_in_euro_cents
0,264929473,2024-10-01 00:08:55,2024-12-02 01:09:26,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:09:10,None,None
1,264929559,2024-10-01 00:13:23,2025-01-01 01:13:51,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:13:42,None,None
2,264929588,2024-10-01 00:14:44,2024-11-12 01:16:14,terminated,PlStreamingPflBellatorWire,173,Wire,2024-10-01 00:16:06,None,None
3,264929939,2024-10-01 00:32:06,2024-10-02 00:35:56,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:32:24,None,None
4,264930257,2024-10-01 00:48:45,2025-01-01 01:49:11,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:49:07,None,None


## 4. Total Revenue

In [38]:
billing_histories['revenue_euro'] = billing_histories['amount_in_euro_cents'] / 100

total_revenue = billing_histories['revenue_euro'].sum()
print('Total Revenue:', total_revenue)

Total Revenue: 4546326.65


## 5. Revenue by Country

In [39]:
revenue_country = billing_histories.groupby('country')['amount_in_euro_cents'].sum() / 100
revenue_country.sort_values(ascending=False)

country
PL    4546326.65
Name: amount_in_euro_cents, dtype: float64

## 6. Active vs Terminated Contracts

In [40]:
contracts['status'] = contracts['terminated_at'].apply(lambda x: 'Active' if pd.isna(x) else 'Terminated')
contracts['status'].value_counts()

status
Terminated    59888
Active        49939
Name: count, dtype: int64

## 7. Merge Billing and Contracts

In [41]:
df = pd.merge(billing_histories, contracts, left_on='contract_id', right_on='id', how='left')
df.head()

,id_x,created_at_x,reason,country,contract_id,status_x,amount_in_euro_cents,product_identifier_x,payout_amount_in_euro_cents,product_type_x,...,created_at_y,terminated_at,state,product_identifier_y,payment_provider_config_profile_id,product_type_y,signed_at,billing_histories_count,billing_histories_sum_in_euro_cents,status_y
0,146630708,2024-10-01 00:13:49,FAILURE: Insufficient Funds,PL,264929559,failed,702.0,PlGameplazaWire,0.0,Wire,...,2024-10-01 00:13:23,2025-01-01 01:13:51,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:13:42,None,None,Terminated
1,146631112,2024-10-01 00:16:10,FAILURE: Charge request rejected due to insuff...,PL,264929588,failed,702.0,PlStreamingPflBellatorWire,0.0,Wire,...,2024-10-01 00:14:44,2024-11-12 01:16:14,terminated,PlStreamingPflBellatorWire,173,Wire,2024-10-01 00:16:06,None,None,Terminated
2,146635427,2024-10-01 00:49:08,None,PL,264930257,ok,702.0,PlGameplazaWire,405.0,Wire,...,2024-10-01 00:48:45,2025-01-01 01:49:11,terminated,PlGameplazaWire,161,Wire,2024-10-01 00:49:07,None,None,Terminated
3,146635883,2024-10-01 00:52:27,None,PL,264930332,ok,863.0,PlProbattlezoneWire,526.0,Wire,...,2024-10-01 00:52:00,2024-11-06 16:00:10,terminated,PlProbattlezoneWire,170,Wire,2024-10-01 00:52:26,None,None,Terminated
4,146636572,2024-10-01 00:57:25,None,PL,264930457,ok,702.0,PlStreamingPflBellatorWire,428.0,Wire,...,2024-10-01 00:57:13,2024-10-14 22:01:11,terminated,PlStreamingPflBellatorWire,173,Wire,2024-10-01 00:57:24,None,None,Terminated


## 8. Product Revenue

In [42]:
product_revenue = df.groupby('product_type_x')['amount_in_euro_cents'].sum() / 100
product_revenue.sort_values(ascending=False)

product_type_x
Wire    4546326.65
Name: amount_in_euro_cents, dtype: float64

## 9. Campaign Performance

In [43]:
campaign = pd.merge(contract_signup_details, contracts, left_on='contract_id', right_on='id')

campaign_perf = campaign.groupby('campaign_name')['contract_id'].count()
campaign_perf.sort_values(ascending=False)

campaign_name
PL Play Gameplaza 2.0    3358
PL Play Gamecraze 2.0    2036
PL Play PFL              1771
Name: contract_id, dtype: int64

## 10. ARPU (Average Revenue Per User)

In [44]:
users = billing_histories.groupby('contract_id')['amount_in_euro_cents'].sum()

arpu = users.mean() / 100
print('ARPU:', arpu)

ARPU: 82.60341309640611


## 11. Daily Revenue Trend

In [45]:
billing_histories['date'] = pd.to_datetime(billing_histories['created_at']).dt.date

daily_revenue = billing_histories.groupby('date')['amount_in_euro_cents'].sum() / 100
daily_revenue.tail()

date
2025-01-27    11129.34
2025-01-28    12630.75
2025-01-29    12343.16
2025-01-30    12177.54
2025-01-31     9713.57
Name: amount_in_euro_cents, dtype: float64

## 12. Churn Rate

In [46]:
terminated = contracts['terminated_at'].notna().sum()
total = len(contracts)

churn_rate = terminated / total * 100
print('Churn Rate:', churn_rate)

Churn Rate: 54.52939623225619


In [47]:
activate.close()